# 🧪 Análise Exploratória de Dados (EDA) — RetailRocket
**Contexto:** Desenvolvimento de um motor de recomendação baseado no comportamento de navegação dos usuários para o Tech Challenge.
**Objetivo:** Compreender a volumetria, cardinalidade, esparsidade e o comportamento temporal do dataset para desenhar as regras de engenharia de recursos e arquitetura da rede neural em PyTorch.

---
## 📦 1. Carregamento Estruturado dos Dados
Nesta etapa, carregamos os arquivos brutos (`events.csv`, `category_tree.csv` e as duas partes de `item_properties`) mapeando seus caminhos relativos de forma isolada da raiz do projeto.

In [2]:
import os
import pandas as pd

# Definir o caminho relativo para a pasta de dados brutos
# Como o notebook está localizado em "notebooks", precisamos subir um nível para acessar a pasta "data"
DATA_RAW_DIR = os.path.join("..", "data", "raw")

print("A iniciar o carregamento dos dados do RetailRocket...\n")

# 1. Carregamento dos Eventos (O core do comporatamento de navegação)
events_path = os.path.join(DATA_RAW_DIR, "events.csv")
if os.path.exists(events_path):
    df_events = pd.read_csv(events_path)
    print(f"Eventos carregados com sucesso!")
    print(f"Dimensão: {df_events.shape[0]:,} linhas e {df_events.shape[1]} colunas")
else:
    print(f"Arquivo de eventos não encontrado em: {events_path}")

# 2. Carregamento da Árvore de Categorias (O core da navegação)
category_path = os.path.join(DATA_RAW_DIR, "category_tree.csv")
if os.path.exists(category_path):
    df_category_tree = pd.read_csv(category_path)
    print(f"Árvore de categoria carregada com sucesso!")
    print(f"Dimensão: {df_category_tree.shape[0]:,} linhas e {df_category_tree.shape[1]} colunas")

# 3. Carregamento e concatenação das Propriedades dos Itens (Partes 1 e 2)
prop1_path = os.path.join(DATA_RAW_DIR, "item_properties_part1.csv")
prop2_path = os.path.join(DATA_RAW_DIR, "item_properties_part2.csv")

if os.path.exists(prop1_path) and os.path.exists(prop2_path):
    print("A carregar as propriedades dos itens (Parte 1 e 2)...")
    df_drop1 = pd.read_csv(prop1_path)
    df_drop2 = pd.read_csv(prop2_path)

    # Concatenar verticalmente as duas partes num único DataFrame funcional
    df_item_properties = pd.concat([df_drop1, df_drop2], ignore_index=True)
    print(f"Propriedades dos itens carregadas e concatenadas com sucesso!")
    print(f"Dimensão: {df_item_properties.shape[0]:,} linhas e {df_item_properties.shape[1]} colunas")

    # Liberar memória RAM apagando os DataFrames temporários que ja não precisamos
    del df_drop1, df_drop2
else:
    print(f"Arquivos de propriedades dos itens não encontrados em: {prop1_path} e {prop2_path}")

print("\nCarregamento dos dados concluído com sucesso!")

A iniciar o carregamento dos dados do RetailRocket...

Eventos carregados com sucesso!
Dimensão: 2,756,101 linhas e 5 colunas
Árvore de categoria carregada com sucesso!
Dimensão: 1,669 linhas e 2 colunas
A carregar as propriedades dos itens (Parte 1 e 2)...
Propriedades dos itens carregadas e concatenadas com sucesso!
Dimensão: 20,275,902 linhas e 4 colunas

Carregamento dos dados concluído com sucesso!


## 📅 2. Tratamento e Engenharia Temporal Inicial
Os registros de tempo nativos do RetailRocket são fornecidos no formato Unix Timestamp (milisegundos). Para viabilizar análises de sazonalidade e comportamento de navegação, realizamos a conversão para objetos legíveis de `datetime`.

In [3]:
# Converter o timestamp para o tipo datetime
df_events['datetime'] = pd.to_datetime(df_events['timestamp'], unit='ms')

# Exibir as primeiras 5 linhas do DataFrame de eventos
df_events.head(5)

,timestamp,visitorid,event,itemid,transactionid,datetime
0,1433221332117,257597,view,355908,NaN,2015-06-02 05:02:12.117
1,1433224214164,992329,view,248676,NaN,2015-06-02 05:50:14.164
2,1433221999827,111016,view,318965,NaN,2015-06-02 05:13:19.827
3,1433221955914,483717,view,253185,NaN,2015-06-02 05:12:35.914
4,1433221337106,951259,view,367447,NaN,2015-06-02 05:02:17.106


## 📊 3. Volumetria do Funil e Cardinalidade de IDs
Para estruturar o pipeline de modelagem, precisamos diagnosticar:
1. O nível de desbalanceamento das interações (Visitas vs. Carrinho vs. Compras).
2. A quantidade de usuários (`visitorid`) e itens (`itemid`) únicos, que determinará a dimensão das camadas de `torch.nn.Embedding` da rede neural.

In [5]:
# Contagem e percentual de eventos por tipo
event_counts = df_events['event']. value_counts()
event_percentages = df_events['event'].value_counts(normalize=True) * 100

# Criar um DataFrame para exibir a contagem e o percentual de eventos
df_funnel = pd.DataFrame({
        'Total de Registros': event_counts,
        'Percentual (%)': event_percentages
})

# Formatando a exibição
df_funnel['Total de Registros'] = df_funnel['Total de Registros'].map('{:,}'.format)
df_funnel['Percentual (%)'] = df_funnel['Percentual (%)'].map('{:.2f}'.format)

df_funnel

,Total de Registros,Percentual (%)
event,,
view,"2,664,312",96.67
addtocart,"69,332",2.52
transaction,"22,457",0.81


## 👤 4. Análise de Distribuição e Cauda Longa (Usuários vs. Itens)
Sistemas de recomendação sofrem severamente com dados esparsos. Nesta célula, avaliamos a frequência de interações por usuário e por produto para identificar o volume de usuários "fantasmas" (com apenas 1 clique) e produtos órfãos, servindo de base para o nosso ponto de corte (*threshold*) no estágio de pré-processamento.

In [6]:
n_users = df_events['visitorid'].nunique()
n_items = df_events['itemid'].nunique()
n_interactions = len(df_events)

print(f"Métricas de Interação:")
print(f"Total de interações no histórico: {n_interactions:,}")
print(f"Quantidade de usuários únicos (visitorid): {n_users:,}")
print(f"Quantidade de produtos únicos (itemid): {n_items:,}")
print(f"Densidade do dataset: {(n_interactions / (n_users * n_items)) * 100:.5f}%")

Métricas de Interação:
Total de interações no histórico: 2,756,101
Quantidade de usuários únicos (visitorid): 1,407,580
Quantidade de produtos únicos (itemid): 235,061
Densidade do dataset: 0.00083%


In [7]:
# Janela temporal do dataset
data_inicio = df_events['datetime'].min()
data_fim = df_events['datetime'].max()
total_dias = (data_fim - data_inicio).days

print(f"Período de Coleta dos Dados:")
print(f"Data Inicial: {data_inicio}")
print(f"Data Final: {data_fim}")
print(f"Amplitude Temporal: {total_dias} dias\n")

# Distribuição por dia da semana (Para entender o comportamento do usuário ao longo da semana)
df_events['day_of_week'] = df_events['datetime'].dt.day_name()
print("Interações por dia da semana:")
print(df_events['day_of_week'].value_counts(normalize=True) * 100)

Período de Coleta dos Dados:
Data Inicial: 2015-05-03 03:00:04.384000
Data Final: 2015-09-18 02:59:47.788000
Amplitude Temporal: 137 dias

Interações por dia da semana:
day_of_week
Tuesday      16.221358
Monday       15.957797
Wednesday    15.642170
Thursday     15.193964
Friday       13.776672
Sunday       12.133880
Saturday     11.074159
Name: proportion, dtype: float64


In [8]:
# Contando quantas interações cada usuário (visitorid) tem no dataset
user_activity = df_events['visitorid'].value_counts()

print(f"Distribuição de atividade por usuário:")
print(f"Média de interações por usuário: {user_activity.mean():.2f}")
print(f"Mediana de interações por usuário: {user_activity.median():.0f}")
print(f"Máximo de interações por usuário: {user_activity.max():,}")

# Quantos usuários têm apenas 1 interação
single_interaction_users = (user_activity == 1).sum()
pct_single_users = (single_interaction_users / len(user_activity)) * 100

print(f"Usuários com apenas 1 interação no histórico:{single_interaction_users:,} ({pct_single_users:.2f}%)")

Distribuição de atividade por usuário:
Média de interações por usuário: 1.96
Mediana de interações por usuário: 1
Máximo de interações por usuário: 7,757
Usuários com apenas 1 interação no histórico:1,001,560 (71.15%)


In [10]:
# Filtrando apenas os eventos de transação
df_transactions = df_events[df_events['event'] == 'transaction']

# Quais são os 10 produtos mais comprados
top_bought_items = df_transactions['itemid'].value_counts().head(10)

print("Top 10 produtos mais comprados no e-commerce:")
print(top_bought_items)

Top 10 produtos mais comprados no e-commerce:
itemid
461686    133
119736     97
213834     92
7943       46
312728     46
445351     45
48030      41
420960     38
248455     38
17478      37
Name: count, dtype: int64


## ⏱️ 5. Dinâmica de Navegação e Padrão de Sessões
Como não há um identificador explícito de sessão no dataset, analisamos o diferencial de tempo (`time_diff`) entre ações consecutivas do mesmo usuário. Essa métrica determinará a janela temporal ideal para agrupar as interações dentro de uma mesma jornada de compra.

In [11]:
# Ordenar o dataset por usuário e data/hora para calcular os intervalos
df_events = df_events.sort_values(by=['visitorid', 'datetime'])

# Calcular o tempo entre interações para cada usuário
df_events['time_diff'] = df_events.groupby('visitorid')['datetime'].diff()
# Converter essa diferença de tempo para minutos
df_events['time_diff_minutes'] = df_events['time_diff'].dt.total_seconds() / 60

print("Estatísticas dos intervalos de tempo entre cliques do mesmo usuário (em minutos)")
print(df_events['time_diff_minutes'].describe())

Estatísticas dos intervalos de tempo entre cliques do mesmo usuário (em minutos)
count    1.348521e+06
mean     3.570517e+03
std      1.506602e+04
min      0.000000e+00
25%      6.372167e-01
50%      2.273467e+00
75%      4.082982e+01
max      1.964575e+05
Name: time_diff_minutes, dtype: float64


## 🏁 Conclusões do EDA para o Pipeline de Produção

Com base nos resultados numéricos obtidos neste rascunho de experimentação, consolidamos as seguintes premissas para os scripts oficiais em `src/stages/`:

1. **Estratégia de Filtragem (`preprocess.py`):** O alto índice de esparsidade justifica a remoção de usuários e itens com baixíssima volumetria de interações para limpar o ruído do modelo.
2. **Abordagem de Target (`feature_eng.py`):** Como as transações ocupam menos de 1% do dataset, adotaremos uma estratégia de **Implicit Feedback**, atribuindo pesos distintos aos eventos (`view`: 1, `addtocart`: 3, `transaction`: 5) para guiar o aprendizado da MLP.
3. **Mapeamento de Embeddings (`train.py`):** Os valores de IDs únicos mapeados serão salvos em dicionários de codificação sequencial densa ($0, 1, 2...$), definindo os hiperparâmetros de entrada das camadas de Embedding do PyTorch.